# QDSV Bridge Colab 01 - Semantic Candidate Marking

This notebook shows the 5-minute Bridge flow:

`Problem spec -> Bridge artifact -> Bridge Report`

The example uses `bounded_semantic_marking`, the controlled fallback family for finite semantic candidate marking.

In [ ]:
!pip -q install -U qdsv-bridge

In [ ]:
import os
from IPython.display import Markdown, display
from qdsv_bridge import QDSVBridgeClient

API_URL = os.getenv("QDSV_BRIDGE_API_URL", "https://qdsv-api-568788785178.us-central1.run.app/api")
API_KEY = os.getenv("QDSV_BRIDGE_API_KEY") or None

client = QDSVBridgeClient(api_url=API_URL, api_key=API_KEY, timeout=60)
families = client.families()
print("API status:", families["status"])
print("Report rate limit:", families["public_limits"]["rate_limits"].get("report"))
print("API key required for:", families["public_limits"].get("api_key_required_for"))

In [ ]:
spec = {
    "family": "bounded_semantic_marking",
    "bridge_mode": "build",
    "state_space": {
        "kind": "finite_candidates",
        "candidate_count": 16,
        "candidate_id": "candidate",
    },
    "signals": ["eligibility_score", "risk_score", "value_score"],
    "goal": {
        "kind": "marking",
        "predicate": "eligible_candidate",
    },
    "target": {
        "format": "qasm3",
        "backend_family": "qiskit",
    },
    "limits": {
        "max_qubits": 6,
        "max_depth": 250,
    },
}

artifact = client.build(spec)
print("status:", artifact["status"])
print("family:", artifact["family"])
print("mode:", artifact["bridge_mode"])
print("artifact format:", artifact["artifact"]["format"])
print("circuit origin:", artifact["circuit_origin"])
print("digests:", artifact["digests"])

print("\nArtifact preview:\n")
print(artifact["artifact"].get("content", "")[:1200])

In [ ]:
report = client.report(spec, mode="build", format="markdown")
display(Markdown(report["content"]))

with open("bridge_report_semantic_candidate_marking.md", "w", encoding="utf-8") as f:
    f.write(report["content"])

print("Saved: bridge_report_semantic_candidate_marking.md")